In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import pandas as pd

load_dotenv()
client = OpenAI()  # Lee OPENAI_API_KEY desde .env

In [2]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("❌ No se encontró la variable OPENAI_API_KEY en el archivo .env")
else:
    print("✅ OPENAI_API_KEY cargada correctamente")

✅ OPENAI_API_KEY cargada correctamente


In [3]:
client = OpenAI()

In [4]:
SYSTEM_PROMPT = """
Eres Hannah 🌸, una QA Engineer Senior especializada en automatización de pruebas.
Tu misión es analizar requerimientos funcionales o historias de usuario (HDU)
y generar:

1. Una matriz de pruebas en formato tabular con columnas:
   | ID | Escenario | Datos de entrada | Resultado esperado |

2. Casos de prueba en lenguaje Gherkin (Cucumber) con el formato:
   Feature: <nombre de la funcionalidad>
     Scenario: <nombre del escenario>
       Given ...
       When ...
       Then ...

📋 Reglas estrictas:
- Los IDs deben ser secuenciales (TC001, TC002, …)
- Mantén los nombres de escenarios concisos pero descriptivos.
- Usa solo español claro y profesional.
- No incluyas texto adicional fuera del formato solicitado.
- Usa comillas dobles (“”) para los mensajes esperados.
- Mantén consistencia entre la matriz y los casos Gherkin.
"""


In [5]:
few_shot_example = """
| ID | Escenario | Datos de entrada | Resultado esperado |
|----|------------|------------------|--------------------|
| TC001 | Login exitoso | Usuario válido y contraseña correcta | Acceso concedido |
| TC002 | Login fallido | Usuario inválido o contraseña incorrecta | Mensaje de error "Credenciales inválidas" |

Feature: Validación de login
  Scenario: Login exitoso
    Given que el usuario ingresa credenciales válidas
    When presiona el botón "Iniciar sesión"
    Then debe mostrarse el mensaje "Acceso concedido"

  Scenario: Login fallido
    Given que el usuario ingresa credenciales incorrectas
    When intenta iniciar sesión
    Then debe mostrarse el mensaje "Credenciales inválidas"
"""

In [6]:
def probar_prompt(requerimiento):
    print("🧩 Analizando requerimiento:\n", requerimiento, "\n")
    
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.3,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"""Aquí tienes un ejemplo de formato esperado:

{few_shot_example}

Ahora genera la matriz de pruebas y los casos Gherkin para el siguiente requerimiento:

{requerimiento}
"""
            }
        ]
    )
    
    output = resp.choices[0].message.content
    print("=== 🧠 SALIDA DE HANNAH ===\n")
    print(output)
    return output

In [7]:
requerimiento = """
[IOS] Generar acceso en menú hamburguesa con la etiqueta nuevo (estrategia de liberación).
Se espera implementar la lógica desagrupada por segmento, iniciando con segmento premier,
creando un acceso en el menú con un flag de LaunchDarkly.
"""

probar_prompt(requerimiento)

🧩 Analizando requerimiento:
 
[IOS] Generar acceso en menú hamburguesa con la etiqueta nuevo (estrategia de liberación).
Se espera implementar la lógica desagrupada por segmento, iniciando con segmento premier,
creando un acceso en el menú con un flag de LaunchDarkly.
 

=== 🧠 SALIDA DE HANNAH ===

| ID    | Escenario                       | Datos de entrada                                     | Resultado esperado                                         |
|-------|---------------------------------|-----------------------------------------------------|-----------------------------------------------------------|
| TC001 | Acceso nuevo en menú hamburguesa | Usuario en segmento premier                          | Debe mostrarse la opción "Nuevo" en el menú hamburguesa   |
| TC002 | Acceso no disponible en menú hamburguesa | Usuario en segmento no premier                       | No debe mostrarse la opción "Nuevo" en el menú hamburguesa |
| TC003 | Comportamiento del flag de LaunchDarkly | U

'| ID    | Escenario                       | Datos de entrada                                     | Resultado esperado                                         |\n|-------|---------------------------------|-----------------------------------------------------|-----------------------------------------------------------|\n| TC001 | Acceso nuevo en menú hamburguesa | Usuario en segmento premier                          | Debe mostrarse la opción "Nuevo" en el menú hamburguesa   |\n| TC002 | Acceso no disponible en menú hamburguesa | Usuario en segmento no premier                       | No debe mostrarse la opción "Nuevo" en el menú hamburguesa |\n| TC003 | Comportamiento del flag de LaunchDarkly | Usuario en segmento premier y flag activado          | Debe mostrarse la opción "Nuevo" en el menú hamburguesa   |\n| TC004 | Comportamiento del flag de LaunchDarkly | Usuario en segmento premier y flag desactivado       | No debe mostrarse la opción "Nuevo" en el menú hamburguesa |\n\nFeature: 